# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face")

# Hugging Face dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

print("Dataset path configured")

DuckDB connected to Hugging Face
Dataset path configured


In [3]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

print(schema["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [4]:
import pandas as pd

fact = f"""
    read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
"""

march_april = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS march_ctr,

        AVG(gsc_avg_position) AS march_avg_position

    FROM {fact}
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks

    FROM {fact}
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.*,
    a.april_impressions,
    a.april_clicks,

    CASE
        WHEN m.march_impressions > 0
             AND a.april_impressions IS NOT NULL
        THEN
            (a.april_impressions - m.march_impressions)
            * 1.0 / m.march_impressions
        ELSE NULL
    END AS impressions_change_pct

FROM march m

INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id

WHERE m.march_impressions > 0
""").df()

print("March → April rows:", len(march_april))
march_april.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March → April rows: 176737


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,april_impressions,april_clicks,impressions_change_pct
0,client_62f4a7e64f5e0096,content_ddbfb1907979759a,13.0,0.0,0.0,5.250000,8.0,0.0,-0.384615
1,client_62f4a7e64f5e0096,content_19d31b32f74b4f12,6.0,0.0,0.0,11.666667,6.0,0.0,0.000000
2,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,13.0,0.0,0.0,30.055556,26.0,0.0,1.000000
3,client_62f4a7e64f5e0096,content_2a44e78f3d53769e,2.0,0.0,0.0,5.000000,17.0,0.0,7.500000
4,client_62f4a7e64f5e0096,content_745efcdf75e0ec8c,13.0,0.0,0.0,6.740741,7.0,0.0,-0.461538


In [7]:
content_schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
""").df()

print(content_schema["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [8]:
freshness_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(content_updated_date) AS updated_date_present,
    COUNT(last_optimized_date) AS optimized_date_present,

    MIN(content_updated_date) AS earliest_updated,
    MAX(content_updated_date) AS latest_updated,

    MIN(last_optimized_date) AS earliest_optimized,
    MAX(last_optimized_date) AS latest_optimized
FROM read_parquet(
    '{rel}/dim_content.parquet'
)
""").df()

freshness_check

,total_rows,updated_date_present,optimized_date_present,earliest_updated,latest_updated,earliest_optimized,latest_optimized
0,519606,519606,45396,2024-10-28,2026-07-06,2026-04-24,2026-07-06


In [9]:
created_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(content_created_date) AS created_date_present,
    MIN(content_created_date) AS earliest_created,
    MAX(content_created_date) AS latest_created
FROM read_parquet(
    '{rel}/dim_content.parquet'
)
""").df()

created_check

,total_rows,created_date_present,earliest_created,latest_created
0,519606,519606,2024-10-16,2026-07-06


In [10]:
volume_buckets = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {fact}
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {fact}
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
),

joined AS (
    SELECT
        m.march_impressions,
        a.april_impressions,
        CASE
            WHEN m.march_impressions BETWEEN 1 AND 9
                THEN '1-9'
            WHEN m.march_impressions BETWEEN 10 AND 49
                THEN '10-49'
            WHEN m.march_impressions BETWEEN 50 AND 199
                THEN '50-199'
            ELSE '200+'
        END AS impression_bucket
    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id
    WHERE m.march_impressions > 0
)

SELECT
    impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(march_impressions), 1) AS avg_march_impressions,
    ROUND(AVG(april_impressions), 1) AS avg_april_impressions,
    ROUND(
        AVG(
            (april_impressions - march_impressions)
            * 1.0 / march_impressions
        ) * 100,
        1
    ) AS avg_change_pct
FROM joined
GROUP BY impression_bucket
ORDER BY
    CASE impression_bucket
        WHEN '1-9' THEN 1
        WHEN '10-49' THEN 2
        WHEN '50-199' THEN 3
        WHEN '200+' THEN 4
    END
""").df()

volume_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_march_impressions,avg_april_impressions,avg_change_pct
0,1-9,33531,3.4,19.9,615.5
1,10-49,27092,25.3,77.9,218.5
2,50-199,31281,110.7,174.5,65.4
3,200+,84833,3258.1,3225.5,6.6


In [11]:
staleness_buckets = con.sql(f"""
WITH content_age AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '2026-03-31'
        ) AS content_age_days
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
    WHERE content_created_date <= DATE '2026-03-31'
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {fact}
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {fact}
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
),

joined AS (
    SELECT
        c.content_age_days,
        m.march_impressions,
        a.april_impressions,
        CASE
            WHEN c.content_age_days < 90
                THEN '<90 days'
            WHEN c.content_age_days < 180
                THEN '90-179 days'
            WHEN c.content_age_days < 365
                THEN '180-364 days'
            ELSE '365+ days'
        END AS age_bucket
    FROM content_age c
    INNER JOIN march m
        ON c.client_hash_id = m.client_hash_id
        AND c.content_hash_id = m.content_hash_id
    INNER JOIN april a
        ON c.client_hash_id = a.client_hash_id
        AND c.content_hash_id = a.content_hash_id
    WHERE m.march_impressions > 0
)

SELECT
    age_bucket,
    COUNT(*) AS n,
    ROUND(AVG(content_age_days), 1) AS avg_age_days,
    ROUND(AVG(march_impressions), 1) AS avg_march_impressions,
    ROUND(AVG(april_impressions), 1) AS avg_april_impressions,
    ROUND(
        AVG(
            (april_impressions - march_impressions)
            * 1.0 / march_impressions
        ) * 100,
        1
    ) AS avg_change_pct
FROM joined
GROUP BY age_bucket
ORDER BY
    CASE age_bucket
        WHEN '<90 days' THEN 1
        WHEN '90-179 days' THEN 2
        WHEN '180-364 days' THEN 3
        WHEN '365+ days' THEN 4
    END
""").df()

staleness_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,age_bucket,n,avg_age_days,avg_march_impressions,avg_april_impressions,avg_change_pct
0,<90 days,57734,47.9,1387.6,1744.1,442.1
1,90-179 days,25872,133.3,2194.9,1748.0,38.2
2,180-364 days,71421,245.7,1539.8,1442.5,36.9
3,365+ days,21710,408.5,1556.3,1516.3,0.8


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks

**Signal 1 — March GSC impressions (volume)**  
**Verdict: CONFIRMED — directional.** Lower-impression buckets showed substantially larger average relative impression growth in April (+615.5% for 1–9 impressions vs +6.6% for 200+). This supports volume as a quick-win screening signal. The result is directional because relative growth is sensitive to small denominators.

**Signal 2 — Content age (staleness)**  
**Verdict: CONFIRMED — directional.** Content younger than 90 days showed +442.1% average relative impression growth, while content 365+ days old showed +0.8%. This supports content age as a useful refresh-screening signal. This is an observed association, not evidence that age itself causes performance changes.

### Baseline rule

I will rank content using two March-only signals: impression volume and content age.

- Low March impressions increase the score because the volume check showed a directional quick-win signal.
- Older content increases the score because the age check showed a directional refresh signal.
- The score is transparent and rule-based; no fitted model weights are used.
- Each content item receives one primary reason code based on the signal contributing the most points.

**Score**
- March impressions < 10 → +2 points
- March impressions 10–49 → +1 point
- March impressions 50+ → +0 points
- Content age 365+ days → +2 points
- Content age 180–364 days → +1 point
- Content age <180 days → +0 points

**Reason codes**
- `QUICK_WIN_VOLUME` — low March impression volume is the strongest reason.
- `REFRESH_STALE` — older content age is the strongest reason.
- `QUICK_WIN_VOLUME` is used when the two signals tie.

**Action labels**
- `Quick win` — prioritize low-volume content for review.
- `Refresh` — prioritize older content for review.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# March 2026 feature window
march_features = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {fact}
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(content_created_date AS DATE) AS content_created_date
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
    WHERE content_created_date <= DATE '2026-03-31'
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-03-31'
    ) AS content_age_days
FROM march m
INNER JOIN content c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id
""").df()

# -----------------------------
# Transparent baseline score
# -----------------------------

# Volume points
march_features["volume_points"] = (
    march_features["march_impressions"]
    .apply(lambda x: 2 if x < 10 else 1 if x < 50 else 0)
)

# Staleness points
march_features["staleness_points"] = (
    march_features["content_age_days"]
    .apply(lambda x: 2 if x >= 365 else 1 if x >= 180 else 0)
)

# Total score
march_features["baseline_score"] = (
    march_features["volume_points"]
    + march_features["staleness_points"]
)

# One primary reason code.
# Volume wins ties, as defined in Section 1.
def assign_reason(row):
    if row["volume_points"] >= row["staleness_points"] and row["volume_points"] > 0:
        return "QUICK_WIN_VOLUME"
    elif row["staleness_points"] > 0:
        return "REFRESH_STALE"
    else:
        return "NO_SIGNAL"

march_features["reason_code"] = march_features.apply(assign_reason, axis=1)

# Action label
def assign_action(reason):
    if reason == "QUICK_WIN_VOLUME":
        return "Quick win"
    elif reason == "REFRESH_STALE":
        return "Refresh"
    else:
        return "No action"

march_features["action"] = march_features["reason_code"].apply(assign_action)

# Rank highest score first.
# Deterministic tie-breakers make the queue reproducible.
ranked_queue = (
    march_features
    .sort_values(
        ["baseline_score", "march_impressions", "content_age_days"],
        ascending=[False, True, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = ranked_queue.index + 1

# Keep the final queue columns simple and auditable
ranked_queue = ranked_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "content_age_days",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

# -----------------------------
# Write required CSV
# -----------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print("Rows ranked:", len(ranked_queue))
print("CSV written:", output_path)

print("\nTop 10:")
display(ranked_queue.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows ranked: 329313
CSV written: work/outputs/baseline_action_score.csv

Top 10:


,rank,client_hash_id,content_hash_id,march_impressions,content_age_days,baseline_score,reason_code,action
0,1,client_625b6439094e23e4,content_006fb87ae2dfc757,0.0,488,4,QUICK_WIN_VOLUME,Quick win
1,2,client_625b6439094e23e4,content_0078ac3847cd9110,0.0,488,4,QUICK_WIN_VOLUME,Quick win
2,3,client_625b6439094e23e4,content_00cc9e2cee3a41ae,0.0,488,4,QUICK_WIN_VOLUME,Quick win
3,4,client_625b6439094e23e4,content_013a5d82d8826884,0.0,488,4,QUICK_WIN_VOLUME,Quick win
4,5,client_625b6439094e23e4,content_01aa5e52698d5a05,0.0,488,4,QUICK_WIN_VOLUME,Quick win
5,6,client_625b6439094e23e4,content_01c3f88bf62e0066,0.0,488,4,QUICK_WIN_VOLUME,Quick win
6,7,client_625b6439094e23e4,content_0305bc72564747c9,0.0,488,4,QUICK_WIN_VOLUME,Quick win
7,8,client_625b6439094e23e4,content_032408cf056b507e,0.0,488,4,QUICK_WIN_VOLUME,Quick win
8,9,client_625b6439094e23e4,content_035cfa22d7e936b0,0.0,488,4,QUICK_WIN_VOLUME,Quick win
9,10,client_625b6439094e23e4,content_04ac8e8a7691bc36,0.0,488,4,QUICK_WIN_VOLUME,Quick win


In [13]:
print("Score distribution:")
display(
    ranked_queue["baseline_score"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("baseline_score")
    .reset_index(name="n")
)

print("\nReason code distribution:")
display(
    ranked_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

Score distribution:


,baseline_score,n
0,4,13136
1,3,131977
2,2,70294
3,1,57155
4,0,56751



Reason code distribution:


,reason_code,n
0,QUICK_WIN_VOLUME,210206
1,REFRESH_STALE,62356
2,NO_SIGNAL,56751


In [14]:
print("Current ranked_queue rows:", len(ranked_queue))
print("Score counts total:", ranked_queue["baseline_score"].value_counts().sum())

display(
    ranked_queue["baseline_score"]
    .value_counts()
    .sort_index(ascending=False)
)

Current ranked_queue rows: 329313
Score counts total: 329313


,count
baseline_score,
4,13136
3,131977
2,70294
1,57155
0,56751


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.